### Notebook to tweek YAIB's preprocessing of the data, to fit a SSL setup
- This means not having labels


#### YAIB preprocessing pipeline

raw data -> split data (funciton) -> preprocess data (class)

In [1]:
import os
import copy
import logging

import gin
import json
import hashlib
import pandas as pd
import polars as pl
from pathlib import Path
import pickle
from timeit import default_timer as timer
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedShuffleSplit, ShuffleSplit
from icu_benchmarks.data.preprocessor import Preprocessor, PandasClassificationPreprocessor, PolarsClassificationPreprocessor
from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys
from icu_benchmarks.data.constants import DataSplit as Split, DataSegment as Segment, VarType as Var


In [2]:
from icu_benchmarks.data.split_process_data import *
from icu_benchmarks.cross_validation import execute_repeated_cv  # adjust if path is different
from icu_benchmarks.run import *
import gin

# Path ti configs
#'os.chdir("/work3/s185395/YAIB/")

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/Regression.gin")


/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


ParsedConfigFileIncludesAndImports(filename='/work3/s185395/YAIB/configs/tasks/Regression.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

In [32]:
import gin

gin.parse_config_file("configs/tasks/Regression.gin")

ParsedConfigFileIncludesAndImports(filename='configs/tasks/Regression.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

In [3]:
# Override the outcome scaling range
gin.bind_parameter("base_regression_preprocessor.outcome_min", 0)
gin.bind_parameter("base_regression_preprocessor.outcome_max", 10)

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("demo_data/los/mimic_demo"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    runmode=RunMode.regression,
)

/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1140: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1145: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1165: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [67]:
data['train'].keys()

dict_keys(['OUTCOME', 'FEATURES'])

#### Next step will try to be to build a dataset class that 
1. takes input like the polardataset classes in YAIB
2. computes the output like the MortalityDataset does it for R's repo 
Then we make sure that the raw datastructures can be preprocessed by YAIB's existing tools and that when the preprocessed data is loaded that it can be inputted into R's implementation 

- Maybe be aware of the paired aspect R did in her work. How is class impalance being handled by YAIB? 

This is what Chat says for now, check conversation again when working 

Polars Input
  → group by stay_id
  → sort by time
  → create (T, F) arrays
  → compute masks
  → compute deltas
  → output (data, times, static, label, mask, delta) per patient
